# Decision Trees from Scratch

In this notebook, we will build a fundamental component of XGBoost: a Decision Tree Regressor.
We will use pure Python and NumPy to understand the mechanics of how a tree learns by minimizing Mean Squared Error (MSE).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Create a Toy Dataset
Let's create a simple 1D dataset with a non-linear relationship.

In [ ]:
# Generate 100 points
X = np.sort(5 * np.random.rand(100, 1), axis=0)
y = np.sin(X).ravel()

# Add some noise
y[::5] += 1 * (0.5 - np.random.rand(20))

plt.scatter(X, y, color="darkorange", label="data")
plt.title("Toy Dataset for Regression")
plt.xlabel("Feature X")
plt.ylabel("Target y")
plt.legend()
plt.show()

## 2. Decision Tree Node
A tree is made of nodes. Each node stores information about the split (feature and threshold) or the final prediction (if it's a leaf node).

In [ ]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        # For decision nodes
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        # For leaf nodes
        self.value = value

    def is_leaf_node(self):
        return self.value is not None

## 3. Decision Tree Regressor
Here we implement the tree. The core logic is in `_best_split`, which iterates over every feature and every possible threshold (value of that feature) to find the split that minimizes the MSE of the resulting child nodes.

In [ ]:
class DecisionTreeRegressorFromScratch:
    def __init__(self, min_samples_split=2, max_depth=100):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X, y):
        self.root = self._grow_tree(X, y, 0)
        
    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        
        # Stopping criteria
        if depth >= self.max_depth or n_samples < self.min_samples_split or len(np.unique(y)) == 1:
            leaf_value = np.mean(y)
            return Node(value=leaf_value)
        
        # Find the best split
        best_feat, best_thresh = self._best_split(X, y)
        
        if best_feat is None:
            # Could not find a valid split
            leaf_value = np.mean(y)
            return Node(value=leaf_value)
            
        # Create children
        left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idxs = np.where(X[:, best_feat] > best_thresh)[0]
        
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)
        
    def _best_split(self, X, y):
        best_mse = float('inf')
        best_feat, best_thresh = None, None
        
        n_samples, n_features = X.shape
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thresh in thresholds:
                left_idxs = np.where(X_column <= thresh)[0]
                right_idxs = np.where(X_column > thresh)[0]
                
                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue
                    
                # Calculate MSE of this split
                mse = self._calculate_split_mse(y, left_idxs, right_idxs)
                
                if mse < best_mse:
                    best_mse = mse
                    best_feat = feat_idx
                    best_thresh = thresh
                    
        return best_feat, best_thresh
        
    def _calculate_split_mse(self, y, left_idxs, right_idxs):
        # The "impurity" of a regression node is its variance (MSE from mean)
        # We want to minimize the weighted variance of the children
        y_left = y[left_idxs]
        y_right = y[right_idxs]
        
        var_left = np.var(y_left) if len(y_left) > 0 else 0
        var_right = np.var(y_right) if len(y_right) > 0 else 0
        
        n_total = len(y)
        mse = (len(y_left) / n_total) * var_left + (len(y_right) / n_total) * var_right
        return mse
        
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
        
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
            
        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

## 4. Train and Visualize
Let's see how our custom tree fits the data.

In [ ]:
# Train our custom tree
tree = DecisionTreeRegressorFromScratch(max_depth=3)
tree.fit(X, y)

# Predict on a dense grid for smooth plotting
X_test = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
y_pred = tree.predict(X_test)

plt.figure()
plt.scatter(X, y, s=20, edgecolor="black", c="darkorange", label="data")
plt.plot(X_test, y_pred, color="cornflowerblue", label="max_depth=3", linewidth=2)
plt.xlabel("data")
plt.ylabel("target")
plt.title("Decision Tree Regression (From Scratch)")
plt.legend()
plt.show()

## 5. Decision Tree Classifier (From Scratch)
We can adapt our Regressor into a Classifier by changing two things:
1. **Split Criterion**: Instead of minimizing Mean Squared Error (MSE), we minimize **Gini Impurity**.
2. **Leaf Node Value**: Instead of the mean of the targets, the leaf predicts the **most common class** (mode).

In [ ]:
from collections import Counter

class DecisionTreeClassifierFromScratch:
    def __init__(self, min_samples_split=2, max_depth=100):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X, y):
        self.root = self._grow_tree(X, y, 0)
        
    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        # Stopping criteria
        if depth >= self.max_depth or n_samples < self.min_samples_split or n_labels == 1:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
        
        best_feat, best_thresh = self._best_split(X, y)
        
        if best_feat is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
            
        left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idxs = np.where(X[:, best_feat] > best_thresh)[0]
        
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)
        
    def _best_split(self, X, y):
        best_gini = float('inf')
        best_feat, best_thresh = None, None
        n_samples, n_features = X.shape
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thresh in thresholds:
                left_idxs = np.where(X_column <= thresh)[0]
                right_idxs = np.where(X_column > thresh)[0]
                
                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue
                    
                gini = self._calculate_gini_split(y, left_idxs, right_idxs)
                
                if gini < best_gini:
                    best_gini = gini
                    best_feat = feat_idx
                    best_thresh = thresh
                    
        return best_feat, best_thresh
        
    def _calculate_gini_split(self, y, left_idxs, right_idxs):
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        
        def _gini(y_subset):
            _, counts = np.unique(y_subset, return_counts=True)
            probabilities = counts / len(y_subset)
            return 1 - np.sum(probabilities ** 2)
            
        gini_l = _gini(y[left_idxs])
        gini_r = _gini(y[right_idxs])
        
        return (n_l / n) * gini_l + (n_r / n) * gini_r
        
    def _most_common_label(self, y):
        counter = Counter(y)
        most_common = counter.most_common(1)[0][0]
        return most_common
        
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
        
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
            
        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

## 6. Testing on the Iris Dataset
Let's load the famous Iris dataset and visualize the decision boundaries using our custom classifier.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
# We use only the first two features (sepal length and width) for easy 2D visualization
X_iris = iris.data[:, :2]
y_iris = iris.target

X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

clf = DecisionTreeClassifierFromScratch(max_depth=4)
clf.fit(X_train, y_train)

predictions = clf.predict(X_test)
accuracy = np.mean(predictions == y_test)
print(f"Accuracy on test set: {accuracy * 100:.2f}%")

# Plotting Decision Boundaries
x_min, x_max = X_iris[:, 0].min() - 0.5, X_iris[:, 0].max() + 0.5
y_min, y_max = X_iris[:, 1].min() - 0.5, X_iris[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, edgecolor="k", cmap=plt.cm.RdYlBu)
plt.title("Decision Boundaries on Iris Dataset (Custom Classifier)")
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.show()